
# Hybrid Multi‑Agent System (ReAct + Coding + Mixture-of-Agents)

Notebook นี้สาธิตระบบ **Multi-Agent แบบ Hybrid** ที่ประกอบด้วย
1) **ReAct (Research) Agent** — ค้นคว้าจาก Wikipedia ด้วย `search_wikipedia(term)`  
2) **Coding Agent** — ให้โมเดลสร้างโค้ด Python, รันจริงด้วย `run_python_code(code)`, และทำ **self‑reflection** เพื่อปรับปรุงผล  
3) **Mixture-of-Agents Orchestrator** — รวมความเห็นจากหลายบทบาท (เช่น `Data Scientist`, `Backend Engineer`) แล้วสังเคราะห์เป็นคำตอบเดียว

> ✅ ตรงตามโจทย์: ใช้ `model_name = "gpt-4o"`, โหลด API key จากไฟล์ `.env` ด้วย `load_dotenv()`, และตั้งค่าไคลเอนต์ด้วย `client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))`

---


In [ ]:

# (Optional) ติดตั้งไลบรารีที่จำเป็นเมื่อรันในเครื่องของคุณเอง
# หากติดตั้งแล้ว ข้ามเซลล์นี้ได้
!pip install python-dotenv openai requests wikipedia matplotlib pandas numpy


In [2]:

import os, io, json, textwrap, time, traceback, sys, ast, contextlib
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI

# --- Environment & Client ---
load_dotenv()
model_name = "gpt-4o"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("OpenAI model:", model_name)
print("API key present:", "Yes" if os.getenv("OPENAI_API_KEY") else "No (set it in your .env)")


OpenAI model: gpt-4o
API key present: Yes



## Wikipedia Utility: `search_wikipedia(term)`
ใช้ MediaWiki API เพื่อดึงสรุปและเนื้อหาบทความจาก Wikipedia ตามคำค้น


In [3]:

def search_wikipedia(term: str, lang: str = "en", max_chars: int = 4000) -> Dict[str, Any]:
    """Fetch summary and page content from Wikipedia.
    Returns a dict: { 'term': str, 'summary': str, 'content': str, 'url': str }
    """
    try:
        # 1) Search for the page
        search_url = f"https://{lang}.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "search",
            "srsearch": term,
            "format": "json"
        }
        r = requests.get(search_url, params=params, timeout=20)
        r.raise_for_status()
        data = r.json()
        hits = data.get("query", {}).get("search", [])
        if not hits:
            return {"term": term, "summary": "", "content": "", "url": ""}
        page_title = hits[0]["title"]

        # 2) Get page content
        content_params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": True,
            "titles": page_title,
            "format": "json",
            "redirects": 1
        }
        r2 = requests.get(search_url, params=content_params, timeout=20)
        r2.raise_for_status()
        d2 = r2.json()
        pages = d2.get("query", {}).get("pages", {})
        page = next(iter(pages.values()))
        extract = page.get("extract", "") or ""

        # 3) Summary endpoint (optional, for concise intro)
        summary_url = f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/{page_title.replace(' ', '_')}"
        rs = requests.get(summary_url, timeout=20)
        summary = ""
        url = ""
        if rs.status_code == 200:
            ds = rs.json()
            summary = ds.get("extract", "") or ""
            url = ds.get("content_urls", {}).get("desktop", {}).get("page", "")

        # 4) Truncate to avoid over-long context
        if len(extract) > max_chars:
            extract = extract[:max_chars] + "...\n[truncated]"

        return {"term": term, "summary": summary, "content": extract, "url": url}
    except Exception as e:
        return {"term": term, "summary": "", "content": f"[search_wikipedia error: {e}]", "url": ""}



## Runtime Utility: `run_python_code(code)`
รันโค้ด Python จริงใน environment เดียวกับ Notebook พร้อมดักจับ stdout/stderr


In [4]:

def run_python_code(code: str) -> Dict[str, Any]:
    """Execute Python code safely and capture stdout/stderr.
    Returns {'ok': bool, 'stdout': str, 'stderr': str, 'error': str|None}
    """
    # Minimal sandbox (note: running arbitrary code can be unsafe; use carefully)
    local_env: Dict[str, Any] = {}
    stdout_buffer, stderr_buffer = io.StringIO(), io.StringIO()

    try:
        compiled = compile(code, filename="<generated>", mode="exec")
        with contextlib.redirect_stdout(stdout_buffer):
            with contextlib.redirect_stderr(stderr_buffer):
                exec(compiled, {"__builtins__": __builtins__, "np": np, "pd": pd, "plt": plt}, local_env)
        return {
            "ok": True,
            "stdout": stdout_buffer.getvalue(),
            "stderr": stderr_buffer.getvalue(),
            "error": None
        }
    except Exception as e:
        return {
            "ok": False,
            "stdout": stdout_buffer.getvalue(),
            "stderr": stderr_buffer.getvalue(),
            "error": traceback.format_exc()
        }



## LLM Helpers
ยูทิลิตี้เล็ก ๆ สำหรับเรียกโมเดล GPT‑4o ผ่าน OpenAI SDK


In [5]:

def llm_chat(messages: List[Dict[str, str]], temperature: float = 0.2, max_tokens: int = 1200) -> str:
    """Call OpenAI Chat Completions with model gpt-4o and return the message content."""
    resp = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content or ""

def llm_json(messages: List[Dict[str, str]], schema_hint: str, temperature: float = 0.0, max_tokens: int = 1200) -> Any:
    """Ask model to return JSON (with a simple system hint)."""
    sys_hint = {
        "role": "system",
        "content": f"Return **only** valid JSON matching this structure: {schema_hint}"
    }
    content = llm_chat([sys_hint] + messages, temperature=temperature, max_tokens=max_tokens)
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # last resort: try to extract JSON from text
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(content[start:end+1])
            except Exception:
                pass
        return {"_raw": content, "_error": "Failed to parse JSON"}



## Research Agent (ReAct‑style lite)
- สกัดคำค้นจากคำถามด้วย LLM  
- ค้น Wikipedia ด้วย `search_wikipedia`  
- สรุปผล/อ้างอิงสั้น ๆ


In [6]:

def research_agent(user_question: str, lang: str = "en") -> Dict[str, Any]:
    # 1) Extract 1-3 search terms
    terms = llm_json(
        [
            {"role": "user", "content": f"Extract 1-3 concise Wikipedia search terms for: {user_question}"}
        ],
        schema_hint='{"terms": ["string", "..."]}'
    )
    term_list = terms.get("terms", [])
    if not term_list:
        term_list = [user_question]

    # 2) Query Wikipedia for each term
    results = [search_wikipedia(t, lang=lang) for t in term_list]

    # 3) Summarize with citations (URLs)
    summary_input = "\n\n".join(
        [f"TERM: {r['term']}\nURL: {r.get('url','')}\nSUMMARY: {r.get('summary','')[:800]}\nCONTENT: {r.get('content','')[:800]}" for r in results]
    )
    synth = llm_chat(
        [
            {"role": "system", "content": "You are a careful research assistant. Concisely summarize key points and provide citations as URLs."},
            {"role": "user", "content": f"Question: {user_question}\n\nSources:\n{summary_input}"}
        ],
        temperature=0.2, max_tokens=700
    )
    return {"question": user_question, "terms": term_list, "raw_sources": results, "summary": synth}



## Coding Agent (with Self‑Reflection)
- สร้างโค้ดด้วย LLM
- รันจริงด้วย `run_python_code`
- ให้ LLM วิเคราะห์ผล (`self‑reflection`) และถ้าจำเป็นปรับโค้ดแล้วรันอีกรอบ (อย่างน้อย 1 รอบ)


In [7]:

@dataclass
class CodeRun:
    code: str
    run: Dict[str, Any]
    reflection: str = ""
    repaired_code: str = ""
    repaired_run: Dict[str, Any] = None

def coding_agent(task_description: str, max_iterations: int = 2) -> CodeRun:
    # 1) Ask for Python code (single file, runnable in notebook)
    code = llm_chat(
        [
            {"role": "system", "content": "Write concise, runnable Python code for a Jupyter notebook. Use matplotlib for charts (no seaborn). Do not set custom colors/styles."},
            {"role": "user", "content": f"Task: {task_description}"}
        ],
        temperature=0.2, max_tokens=1400
    )

    # Extract fenced code if present
    def extract_code(text: str) -> str:
        if "```" in text:
            blocks = text.split("```")
            # find python block
            for i in range(1, len(blocks), 2):
                header = blocks[i].strip().split("\n", 1)[0].lower()
                body = blocks[i].split("\n", 1)[1] if "\n" in blocks[i] else ""
                if "python" in header or header == "":
                    return body
        return text

    code_snippet = extract_code(code)
    first_run = run_python_code(code_snippet)

    # 2) Self‑reflection
    reflection = llm_chat(
        [
            {"role": "system", "content": "You are a senior Python reviewer. Diagnose problems based on stdout/stderr and propose minimal fixes."},
            {"role": "user", "content": f"Task: {task_description}\n\nCODE:\n{code_snippet}\n\nSTDOUT:\n{first_run['stdout']}\n\nSTDERR/ERROR:\n{first_run['stderr']}\n{first_run['error'] or ''}\n\nGive a short reflection and a fixed code if needed."}
        ],
        temperature=0.2, max_tokens=900
    )

    repaired_code = ""
    repaired_run = None

    # Try to pull a repaired code block, if any
    extracted = reflection
    if "```" in extracted:
        blocks = extracted.split("```")
        for i in range(1, len(blocks), 2):
            header = blocks[i].strip().split("\n", 1)[0].lower()
            body = blocks[i].split("\n", 1)[1] if "\n" in blocks[i] else ""
            if "python" in header or header == "":
                repaired_code = body
                break

    if repaired_code and repaired_code.strip() != code_snippet.strip():
        repaired_run = run_python_code(repaired_code)

    return CodeRun(
        code=code_snippet,
        run=first_run,
        reflection=reflection,
        repaired_code=repaired_code,
        repaired_run=repaired_run
    )



## Mixture‑of‑Agents Orchestrator
ให้แต่ละบทบาท (เช่น `Epidemiologist`, `Data Analyst`) สร้างข้อเสนอของตนเอง แล้วรวมผลเป็นคำตอบเดียว


In [8]:

def mixture_of_agents(user_prompt: str, agent_roles: List[str]) -> Dict[str, Any]:
    proposals = []
    for role in agent_roles:
        proposal = llm_chat(
            [
                {"role": "system", "content": f"You are acting as a {role}. Provide a clear, structured proposal addressing the user's prompt."},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.4, max_tokens=900
        )
        proposals.append({"role": role, "content": proposal})

    # Aggregate / Synthesize
    aggregate = llm_chat(
        [
            {"role": "system", "content": "Synthesize multiple expert proposals into a single, coherent plan with bullet points and clear rationale."},
            {"role": "user", "content": json.dumps(proposals, ensure_ascii=False)}
        ],
        temperature=0.2, max_tokens=900
    )
    return {"roles": agent_roles, "proposals": proposals, "final": aggregate}



## End-to-End Orchestrator: `multi_agent_system(question)`
พายป์ไลน์ง่าย ๆ:
1) Research Agent — ค้น Wikipedia
2) Coding Agent — ผลิต/รันโค้ดที่เกี่ยวข้อง
3) Mixture‑of‑Agents — รวมข้อเสนอจากหลายบทบาท


In [13]:

def multi_agent_system(question: str, roles: List[str] = None, lang: str = "th") -> Dict[str, Any]:
    if roles is None:
        # ให้โมเดลเลือกบทบาท 2 ตัวอัตโนมัติ
        j = llm_json(
            [{"role": "user", "content": f"Pick 2 concise expert roles that best address this prompt: {question}"}],
            schema_hint='{"roles": ["string", "string"]}'
        )
        roles = j.get("roles", ["Data Scientist", "Backend Engineer"])

    research = research_agent(question, lang=lang)

    # อธิบายให้ Coding Agent ทำงานตามโจทย์ (Code-first)
    coding_task = f"""
    Based on this user question, write Python code to (if applicable) fetch/process data and produce the requested charts/tables.
    Use only matplotlib (no seaborn). Do not set custom colors or styles.
    If internet data is needed (e.g., Wikipedia or CSV), include requests/processing code and handle errors gracefully.
    Question: {question}
    """
    code_result = coding_agent(coding_task, max_iterations=2)

    mix = mixture_of_agents(question, roles)
    return {"research": research, "code": code_result, "mixture": mix, "roles": roles}



## ชุดคำถามตัวอย่าง (จากโจทย์)
> หมายเหตุ: การรันจริงจะเรียกใช้ API และอินเทอร์เน็ต โปรดตรวจสอบให้พร้อมก่อนรัน


In [14]:

questions = [
    # 1. หัวข้อ AI เบื้องต้น:
    "สรุปแนวคิดหลักของ 'Reinforcement Learning' จาก Wikipedia แล้วสร้างกราฟแสดง reward function ด้วย Python พร้อมสะท้อนผลลัพธ์ทุกขั้นตอน",

    # # 2. วิเคราะห์ข้อมูลจริง:
    # "ใช้ Research Agent ดึงข้อมูลสถิติ COVID-19 รายวันจาก Wikipedia จากนั้น Coding Agent คำนวณค่า moving average ของยอดผู้ติดเชื้อ 7 วัน และ Mixture-of-Agents ให้บทบาท 'Epidemiologist' และ 'Data Analyst' เสนอแนวทางจัดการข้อมูล",

    # # 3. เปรียบเทียบอัลกอริทึม:
    # "เปรียบเทียบเวลาในการรันอัลกอริทึม 'Quick Sort' และ 'Merge Sort' ด้วย Coding Agent พร้อม Self-Reflection ว่าการเลือกใช้โครงสร้างข้อมูลใดเหมาะสมกว่า และให้ Mixture-of-Agents สะท้อนมุมมองของ 'Algorithm Theorist' กับ 'Benchmark Specialist'",

    # # 4. ออกแบบ Chatbot ง่าย ๆ:
    # "สร้างระบบ Chatbot เบื้องต้น: Research Agent ค้นหา FAQ จาก Wikipedia, Coding Agent เขียนโค้ด Flask ให้สามารถรับข้อความและตอบกลับ, Mixture-of-Agents ให้บทบาท 'UX Designer' และ 'Backend Engineer' วิเคราะห์การทำงาน และสะท้อนปรับปรุง UX/UI",

    # # 5. วิเคราะห์ชุดข้อมูล Titanic:
    # "Research Agent สรุปที่มาของชุดข้อมูล Titanic จาก Wikipedia, Coding Agent โหลดข้อมูลและคำนวณอัตราการรอดชีวิตตามเพศและชั้นผู้โดยสาร พร้อม self-reflection, Mixture-of-Agents ให้บทบาท 'Statistician' และ 'Data Engineer' สรุปผลเชิงตีความ"
]
questions


["สรุปแนวคิดหลักของ 'Reinforcement Learning' จาก Wikipedia แล้วสร้างกราฟแสดง reward function ด้วย Python พร้อมสะท้อนผลลัพธ์ทุกขั้นตอน"]


## วิธีรันทดสอบ
รันทีละข้อ:
```python
q = questions[0]
result = multi_agent_system(q)                   # รวม research + code + mixture-of-agents
final  = mixture_of_agents(q, ["Role1", "Role2"]) # ถ้าต้องการทดสอบฟังก์ชันนี้โดยตรง

# แสดงผลลัพธ์บางส่วน
result["research"]["summary"]
result["code"].run, result["code"].reflection
result["mixture"]["final"]
```
> คุณสามารถเปลี่ยนบทบาทได้ เช่น `["Epidemiologist", "Data Analyst"]`


In [15]:

def show_research(res):
    print("=== Research Summary ===")
    print(res["research"]["summary"][:2000])
    print("\nSources:")
    for s in res["research"]["raw_sources"]:
        if s.get("url"):
            print("-", s["url"])

def show_code(res):
    c = res["code"]
    print("=== Generated Code ===\n")
    print(c.code)
    print("\n--- First Run ---")
    print("OK:", c.run["ok"])
    print("STDOUT:\n", c.run["stdout"][:2000])
    if c.run["error"]:
        print("\nERROR:\n", c.run["error"][:2000])

    print("\n--- Reflection ---\n", c.reflection[:2000])

    if c.repaired_code:
        print("\n=== Repaired Code ===\n")
        print(c.repaired_code)
        if c.repaired_run:
            print("\n--- Repaired Run ---")
            print("OK:", c.repaired_run["ok"])
            print("STDOUT:\n", c.repaired_run["stdout"][:2000])
            if c.repaired_run["error"]:
                print("\nERROR:\n", c.repaired_run["error"][:2000])

def show_mixture(res):
    print("=== Roles ===", res["roles"])
    print("\n=== Final Synthesis ===\n")
    print(res["mixture"]["final"][:3000])


In [18]:

# ตัวอย่างการรันเร็ว ๆ (อย่าลืมตั้งค่า .env ก่อนและมีอินเทอร์เน็ต)
q = questions[0]
res = multi_agent_system(q)
show_research(res)
show_code(res)
show_mixture(res)


=== Research Summary ===
### แนวคิดหลักของ Reinforcement Learning

Reinforcement Learning (RL) เป็นสาขาหนึ่งของการเรียนรู้ของเครื่อง (Machine Learning) ที่มุ่งเน้นการเรียนรู้ผ่านการกระทำและผลตอบแทน (reward) โดยมีองค์ประกอบหลักดังนี้:

1. **Agent**: ตัวแทนที่ทำการเรียนรู้และตัดสินใจ
2. **Environment**: สภาพแวดล้อมที่ Agent ทำการโต้ตอบ
3. **Actions**: การกระทำที่ Agent สามารถเลือกทำได้
4. **States**: สถานะต่าง ๆ ของ Environment ที่ Agent สามารถรับรู้ได้
5. **Reward**: ค่าตอบแทนที่ Agent ได้รับหลังจากทำการกระทำหนึ่ง ๆ ซึ่งใช้ในการปรับปรุงการตัดสินใจในอนาคต

RL มุ่งเน้นให้ Agent เรียนรู้วิธีการเลือกการกระทำที่เพิ่มผลตอบแทนสะสมสูงสุดในระยะยาว โดยใช้กลยุทธ์การสำรวจ (exploration) และการใช้ประโยชน์ (exploitation) เพื่อปรับปรุงการตัดสินใจ

แหล่งข้อมูล: [Reinforcement Learning - Wikipedia](https://th.wikipedia.org/wiki/%E0%B8%AD%E0%B8%A0%E0%B8%B4%E0%B8%98%E0%B8%B2%E0%B8%99%E0%B8%A8%E0%B8%B1%E0%B8%9E%E0%B8%97%E0%B9%8C%E0%B8%9B%E0%B8%B1%E0%B8%8D%E0%B8%8D%E0%B8%B2%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B8%9